In [2]:
import random

## Game Parameters

In [8]:
# How many points are given on each turn
both_coop = 3
i_defect_you_coop = 5
i_coop_you_defect = 0
both_defect = 1
total_turns = 200
coop_fail_rate = 0.0

In [ ]:
# 1 = coop
# 0 = defect

## Game State Object

In [17]:
class GameState:
    def __init__(self, player_A_strat, player_B_strat, num_turns, coop_noise, turns_known=False):
        self.player_A_strategy = player_A_strat
        self.player_B_strategy = player_B_strat
        self.player_A_move_history = []
        self.player_B_move_history = []
        self.player_A_points = 0
        self.player_B_points = 0
        self.total_turns = num_turns
        self.turns_known = turns_known
        self.coop_fail_rate = coop_noise
        self.curr_turns = 0

    def advance_turn(self):
        # Get initial move from strategies
        player_A_move = self.player_A_strategy(self.player_A_move_history, self.player_B_move_history, self.turns_known)
        player_B_move = self.player_B_strategy(self.player_B_move_history, self.player_A_move_history, self.turns_known)
        # Apply randomness (if applicable)
        if coop_fail_rate > 0:
            if ((player_A_move == 1) and (random.random() < coop_fail_rate)):
                player_A_move = 0
            if ((player_B_move == 1) and (random.random() < coop_fail_rate)):
                player_B_move = 0
        # Add moves to move history
        self.player_A_move_history.append(player_A_move)
        self.player_B_move_history.append(player_B_move)
        # Add points to point totals
        if ((player_A_move == 1) and (player_B_move == 1)):
            self.player_A_points += both_coop
            self.player_B_points += both_coop
        elif ((player_A_move == 1) and (player_B_move == 0)):
            self.player_A_points += i_coop_you_defect
            self.player_B_points += i_defect_you_coop
        elif ((player_A_move == 0) and (player_B_move == 1)):
            self.player_A_points += i_defect_you_coop
            self.player_B_points += i_coop_you_defect
        elif ((player_A_move == 0) and (player_B_move == 0)):
            self.player_A_points += both_defect
            self.player_B_points += both_defect
        # Advance turn count
        self.curr_turns += 1

    def run_game(self):#, player_A_strat, player_B_strat, num_turns, coop_noise, turns_known=False):
        #self.__init__(self, player_A_strat, player_B_strat, num_turns, coop_noise, turns_known)
        while self.curr_turns < self.total_turns:
            self.advance_turn()
        # Report out winner when done
        print('Player A', self.player_A_strategy, ':', self.player_A_points, 'points')
        print('Player B', self.player_B_strategy, ':', self.player_B_points, 'points')
        #if self.player_A_points > self.player_B_points:
        #    print('Plauer A, 'self.player_A_strategy), 'wins!')
        
        

## Stategies

In [400]:
def tit_for_tat (my_moves, opp_moves, total_turns_known):
    # Nice
    if len(opp_moves) == 0:
        return 1
    return opp_moves[-1]

def friedman (my_moves, opp_moves, total_turns_known):
    opp_has_defected = False
    # Nice
    if len(opp_moves) == 0:
        return 1
    # Hold grudge
    if opp_moves[-1] == 0:
        opp_has_defected = True
    if opp_has_defected:
        return 0
    else:
        return 1
def joss (my_moves, opp_moves, total_turns_known):
    sneaky_fraction = 0.10
    # Nice
    if len(opp_moves) == 0:
        return 1
    next_move = opp_moves[-1]
    if next_move == 1:
        if random.random() < sneaky_fraction:
            next_move = 0
    return next_move

def grasskamp (my_moves, opp_moves, total_turns_known):
    sneaky_fraction = 0.10
    score_not_good_threshold = 150
    if len(opp_moves) >= 56:

        my_score = 0
        opp_score = 0
        
        for j in range(0, 56):
            if ((my_moves[j] == 1) and (opp_moves[j] == 1)):
                my_score += both_coop
                opp_score += both_coop
            elif ((my_moves[j] == 1) and (opp_moves[j] == 0)):
                my_score += i_coop_you_defect
                opp_score += i_defect_you_coop
            elif ((my_moves[j] == 0) and (opp_moves[j] == 1)):
                my_score += i_defect_you_coop
                opp_score += i_coop_you_defect
            elif ((my_moves[j] == 0) and (opp_moves[j] == 0)):
                my_score += both_defect
                opp_score += both_defect
        
        grasskamp_pattern = [*50*[1],*6*[0]]
        tit_for_tat_pattern = [*50*[1],*[1,0,1,0,1,0]]

        if opp_moves[:56] == grasskamp_pattern:
            end_strategy = 'always_coop'
        elif opp_moves[:56] == tit_for_tat_pattern:
            end_strategy = 'tit_for_tat'
        elif my_score < score_not_good_threshold:
            end_strategy = 'always_defect'
        else:
            end_strategy = None
    
    # Nice
    if len(opp_moves) == 0:
        return 1
    # Tit for Tat for first 50 moves
    if len(opp_moves) < 50:
        return opp_moves[-1]
    # Nasty Probing on move 51
    if len(opp_moves) == 50:
        return 0
    # Tit for Tat for next 5 moves
    if len(opp_moves) < 56:
        return opp_moves[-1]

    # if len(opp_moves) == 56:
    #     # Now check history of game to see if we recognize the strategy
    #     # Do I make this stochastic, or what? 
    #     # I think I don't want to make this resiliant to noise
    #     grasskamp_pattern = [*50*[1],*6*[0]]
    #     tit_for_tat_pattern = [*50*[1],*[1,0,1,0,1,0]]
    #     # internal scorekeeping
    #     my_score = 0
    #     opp_score = 0
    
    #     # THIS ASSUMES WE KNOW THE RULES OF THE GAME IN TERMS OF WIN AND LOSS SCORING
    #     for j in range(0, len(opp_moves)):
    #         if ((my_moves[j] == 1) and (opp_moves[j] == 1)):
    #             my_score += both_coop
    #             opp_score += both_coop
    #         elif ((my_moves[j] == 1) and (opp_moves[j] == 0)):
    #             my_score += i_coop_you_defect
    #             opp_score += i_defect_you_coop
    #         elif ((my_moves[j] == 0) and (opp_moves[j] == 1)):
    #             my_score += i_defect_you_coop
    #             opp_score += i_coop_you_defect
    #         elif ((my_moves[j] == 0) and (opp_moves[j] == 0)):
    #             my_score += both_defect
    #             opp_score += both_defect
        
    #     if opp_moves == grasskamp_pattern:
    #         end_strategy = 'always_coop'
    #     elif opp_moves == tit_for_tat_pattern:
    #         end_strategy = 'tit_for_tat'
    #     elif my_score < score_not_good_threshold:
    #         end_strategy = 'always_defect'

    #     print(end_strategy)

    if len(opp_moves) >= 56:
        if end_strategy == 'always_coop':
            return 1
        elif end_strategy == 'tit_for_tat':
            #print('opp_moves[-1]:', opp_moves[-1])
            return opp_moves[-1]
        elif end_strategy == 'always_defect':
            return 0
        else: 
            #print('inside else block for Joss!')
            # Paper says defect every 10 to 15 moves, so I'm just using Joss here...
            next_move = opp_moves[-1]
            if next_move == 1:
                if random.random() < sneaky_fraction:
                    next_move = 0
            return next_move

def shubik (my_moves, opp_moves, total_turns_known):
    # Nice
    if len(opp_moves) == 0:
        return 1
    # Check if actively defecting
    if opp_moves[-1] == 0:
        return 0
    if len(opp_moves) == 1:
        return opp_moves[-1]
    # Check if we need to keep exacting revenge
    num_coop_runs = 0
    inside_coop_run = False
    for j in range(0,len(my_moves)-1):
        if not inside_coop_run:
            if my_moves[j] == 1 and my_moves[j+1] == 1:
                num_coop_runs += 1
                inside_coop_run = True
        else: # so inside_coop_run = True
            if my_moves[j] == 0:
                inside_coop_run = False
    #print('Total coop runs:', num_coop_runs)

    # if we're in a good run, then be good
    if ((my_moves[-1] == 1) and (opp_moves[-1] == 1)):
        return 1
    #look back the number of your own moves equal to the num_coop_runs
    for k in range(1, num_coop_runs+1):
        #print(my_moves[-k])
        if my_moves[-k] == 1:
            return 0
    return 1
                

def feld (my_moves, opp_moves, total_turns_known):
    sneaky_fraction = len(opp_moves)/400 # gives us 0% at the beginning, and 50% at the end, turn 200
    # Nice
    if len(opp_moves) == 0:
        return 1
    next_move = opp_moves[-1]
    if next_move == 1:
        if random.random() < sneaky_fraction:
            next_move = 0
    return next_move

def tullock (my_moves, opp_moves, total_turns_known):
    lookback_window = 10
    # Cooperate for first 11 moves
    if len(opp_moves) <= 11:
        return 1
    recent_defections_count = 0
    p_opp_coop = (sum(opp_moves[-lookback_window:])/lookback_window)
    p_my_coop = p_opp_coop - 0.1
    if random.random() < p_my_coop:
        return 1
    return 0

def grofman (my_moves, opp_moves, total_turns_known):
    # Nice
    if len(opp_moves) == 0:
        return 1
    if opp_moves[-1] != my_moves[-1]:
        if random.random() > (2/7):
            return 1
        return 0
    return 1

def davis_and_cuny (my_moves, opp_moves, total_turns_known):
    opp_has_defected = False
    # Cooperate for first 10 moves
    if len(opp_moves) <= 10:
        return 1
    # Plays friedman for the rest of the game
    if opp_moves[-1] == 0:
        opp_has_defected = True
    if opp_has_defected:
        return 0
    else:
        return 1

def downing (my_moves, opp_moves, total_turns_known):
    tiebreaker = 1 # if expected values are identical, then cooperate
    p_coop_if_i_coop = 0.5
    p_coop_if_i_defect = 0.5

    if len(opp_moves) >= 2:
        #print('inside if block')
        num_coops_when_i_cooped = 0
        num_coops_when_i_defected = 0
        for i in range(1, len(opp_moves)):
            if opp_moves[i] == 1:
                if my_moves[i-1] == 1:
                    num_coops_when_i_cooped += 1
                elif my_moves[i-1] == 0:
                    num_coops_when_i_defected += 1
        num_times_i_cooped = 0
        num_times_i_defected = 0
        for j in range(0, len(my_moves)):
            if my_moves[j] == 1:
                num_times_i_cooped += 1
            elif my_moves[j] == 0:
                num_times_i_defected += 1
        if num_times_i_cooped > 0:
            p_coop_if_i_coop = num_coops_when_i_cooped/(len(opp_moves)-1)
        if num_times_i_defected > 0:
            p_coop_if_i_defect = num_coops_when_i_defected/(len(opp_moves)-1)

        #print('p_coop_if_i_coop:', p_coop_if_i_coop)
        #print('p_coop_if_i_defect:', p_coop_if_i_defect)
    
    expected_value_if_i_coop = (p_coop_if_i_coop * both_coop) + ((1-p_coop_if_i_coop) * i_coop_you_defect)
    expected_value_if_i_defect = (p_coop_if_i_defect * i_defect_you_coop) + ((1-p_coop_if_i_defect) * both_defect)
    if expected_value_if_i_coop > expected_value_if_i_defect:
        return 1
    elif expected_value_if_i_coop < expected_value_if_i_defect:
        return 0
    else:
        return tiebreaker

def nydegger (my_moves, opp_moves, total_turns_known):
    if len(opp_moves) == 0:
        return 1
    if len(opp_moves) == 1:
        return opp_moves[-1]
    if len(opp_moves) == 2: # third move
        if ((my_moves[0] == 1) and (opp_moves[0] == 0) and (my_moves[1] == 0) and (opp_moves[1] == 1)):
            return 0
        return opp_moves[-1]
    if len(opp_moves) >= 3:
        A = 0
        if opp_moves[-3] == 0:
            A += (16 * 2)
        if opp_moves[-2] == 0:
            A += (4 * 2)
        if opp_moves[-1] == 0:
            A += (1 * 2)
        if my_moves[-3] == 0:
            A += (16 * 1)
        if my_moves[-2] == 0:
            A += (4 * 1)
        if my_moves[-1] == 0:
            A += (1 * 1)
    if A in [1, 6, 7, 17, 22, 23, 26, 29, 30, 31, 33, 38, 39, 45, 49, 54, 55, 58, 61]:
        return 0
    return 1

# NEED TO FINISH THIS ONE NEED TO INTEGRATE SHUBIK LOGIC FOR INCREASED RETALIATIONS SOMEHOW
# Doing these strategies stateless was a mistake, smh. 
def tideman_and_chieruzzi (my_moves, opp_moves, total_turns_known):
    # function for statistical checking
    def is_three_sigma_dev(defects, total_moves):
        expected_defects = total_moves / 2
        three_sigma = 1.5 * (total_moves ** 0.5)
        deviation = abs(defects - expected_defects)
        return deviation > three_sigma

    # Automatically defect on the last two turns
    if ((len(opp_moves) == 199) or (len(opp_moves) == 198)):
        return 0
    
    in_fresh_start = False
    last_fresh_start_index = 0


    # At this point, I think I need to iterate through all of the different moves in the move history to effectively regenerate the state
    for k in range(1, len(opp_moves)+1):
        opp_moves_temp = opp_moves[0:k]
        my_moves_temp = my_moves[0:k]
    
        # internal scorekeeping
        my_score = 0
        opp_score = 0

        # THIS ASSUMES WE KNOW THE RULES OF THE GAME IN TERMS OF WIN AND LOSS SCORING
        for j in range(0, len(opp_moves_temp)):
            if ((my_moves_temp[j] == 1) and (opp_moves_temp[j] == 1)):
                my_score += both_coop
                opp_score += both_coop
            elif ((my_moves_temp[j] == 1) and (opp_moves_temp[j] == 0)):
                my_score += i_coop_you_defect
                opp_score += i_defect_you_coop
            elif ((my_moves_temp[j] == 0) and (opp_moves_temp[j] == 1)):
                my_score += i_defect_you_coop
                opp_score += i_coop_you_defect
            elif ((my_moves_temp[j] == 0) and (opp_moves_temp[j] == 0)):
                my_score += both_defect
                opp_score += both_defect
    
        # check fresh start conditions
        # opponent is 10 or more points behind
        condition_1 = opp_score <= my_score - 10
        # oppoent has not just started a run of defections
        try:
            condition_2 = not ((opp_moves_temp[-1] == 0) and (opp_moves_temp[-2] == 0))
        except:
            if len(opp_moves_temp) < 2:
                condition_2 = False
            else:
                condition_2 = not ((opp_moves_temp[-1] == 0) and (opp_moves_temp[-2] == 0))
        # more than 20 moves since last fresh start
        condition_3 = len(opp_moves_temp) - last_fresh_start_index >= 20
        # at least 10 moves remaining
        condition_4 = len(opp_moves_temp) <= 190 # could be off by one here...
        # and defections differs from 50/50 generator by more than 3 sigma
        num_opp_defects = 0
        for j in range(0, len(opp_moves_temp)):
            if opp_moves_temp[j] == 0:
                num_opp_defects += 1
        condition_5 = is_three_sigma_dev(num_opp_defects, len(opp_moves_temp))

        if condition_1 and condition_2 and condition_3 and condition_4 and condition_5:
            in_fresh_start = True
            last_fresh_start_index = k
            #print('last fresh start index is updated to:', k)

        if k > last_fresh_start_index + 1:
            in_fresh_start = False

    if in_fresh_start:
        return 1

    #print('shubik move being played!')
    return shubik(my_moves[last_fresh_start_index:], opp_moves[last_fresh_start_index:], total_turns_known)
        
    
def stein_and_rapoport (my_moves, opp_moves, total_turns_known):
    
    if len(opp_moves) >= 15:
        how_far_out = (len(opp_moves)//15) * 15
        if (len(opp_moves[:how_far_out]) % 15 == 0):
        # count each transition
            cc_count = 0
            cd_count = 0
            dc_count = 0
            dd_count = 0
            
            for j in range(0, len(opp_moves)-1):
                if ((opp_moves[j] == 1) and (opp_moves[j+1] == 1)):
                    cc_count += 1
                elif ((opp_moves[j] == 1) and (opp_moves[j+1] == 0)):
                    cd_count += 1
                elif ((opp_moves[j] == 0) and (opp_moves[j+1] == 1)):
                    dc_count += 1
                elif ((opp_moves[j] == 0) and (opp_moves[j+1] == 0)):
                    dd_count += 1
            total = len(opp_moves)-1
            cc_frac = cc_count/total
            cd_frac = cd_count/total
            dc_frac = dc_count/total
            dd_frac = dd_count/total
            chi_sq = 0
            exp_count = total/4
            chi_sq = (((cc_count - exp_count)**2)/exp_count) + (((cd_count - exp_count)**2)/exp_count) + (((dc_count - exp_count)**2)/exp_count) + (((dd_count - exp_count)**2)/exp_count)
            #print('Chi squared:', chi_sq)
            if chi_sq < 7.815: # 95% confidence opponent is playing randomly, so we assume random here
                # unless, there is a run of CDCDCDCDCD or DCDCDCDCDC, in which case we assume some memeroy, then we assume some memory, and just play tit for tat
                if ((opp_moves[-10:] == [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]) or (opp_moves[-10:] == [0, 1, 0, 1, 0, 1, 0, 1, 0, 1])):
                    #print('detecting oscilation, playing tit for tat!')
                    return opp_moves[-1]
                # but if not, if no alternating pattern then we already proved randomness, so we just defect here
                #print('detecting as random, defecting!')
                return 0
    
    # Always coop on first 4 rounds
    elif len(opp_moves) < 4:
        return 1
    # Always defect on the last 2 rounds
    if ((len(opp_moves) == 199) or (len(opp_moves) == 198)):
        return 0
    # all ense fails, we just do tit for tat
    return opp_moves[-1]

def name_witheld (my_moves, opp_moves, total_turns_known):
    # inital prob coop
    p_coop = 0.30
    # need to iterate through the move history to get the most recently determined prob coop
    for k in range(1, len(opp_moves)+1):
        opp_moves_temp = opp_moves[:k]
        my_moves_temp = my_moves[:k]
        
        if (len(opp_moves_temp) % 10 == 0) and (len(opp_moves_temp) > 2):
            #update proability
            num_coops_when_i_cooped = 0
            #num_coops_when_i_defected = 0
            for i in range(1, len(opp_moves_temp)):
                if opp_moves_temp[i] == 1:
                    if my_moves_temp[i-1] == 1:
                        num_coops_when_i_cooped += 1
            num_times_i_cooped = 0
            #num_times_i_defected = 0
            for j in range(0, len(my_moves_temp)):
                if my_moves_temp[j] == 1:
                    num_times_i_cooped += 1
        
            if num_times_i_cooped > 0:
                p_coop_if_i_coop = num_coops_when_i_cooped/(len(opp_moves_temp)-1)

            if p_coop_if_i_coop >= 0.7:
                other_player_type = 'very_coop'
                p_coop += 0.1
            elif p_coop_if_i_coop <= 0.3:
                other_player_type = 'very_uncoop'
                p_coop -= 0.1
            else:
                pass

        # adjust probability at move 130 if we're losing
        if (len(opp_moves_temp) % 130 == 0): 
            # internal scorekeeping
            my_score = 0
            opp_score = 0
    
            # THIS ASSUMES WE KNOW THE RULES OF THE GAME IN TERMS OF WIN AND LOSS SCORING
            for j in range(0, len(opp_moves_temp)):
                if ((my_moves_temp[j] == 1) and (opp_moves_temp[j] == 1)):
                    my_score += both_coop
                    opp_score += both_coop
                elif ((my_moves_temp[j] == 1) and (opp_moves_temp[j] == 0)):
                    my_score += i_coop_you_defect
                    opp_score += i_defect_you_coop
                elif ((my_moves_temp[j] == 0) and (opp_moves_temp[j] == 1)):
                    my_score += i_defect_you_coop
                    opp_score += i_coop_you_defect
                elif ((my_moves_temp[j] == 0) and (opp_moves_temp[j] == 0)):
                    my_score += both_defect
                    opp_score += both_defect
                    
            if my_score <  opp_score:
                p_coop -= 0.10

    if random.random() < p_coop:
        return 0
    return 1
    
def random_strat (my_moves, opp_moves, total_turns_known):
    if random.random() < 0.5:
        return 0
    return 1

### Test Strategies
def random_low_chance (my_moves, opp_moves, total_turns_known):
    if random.random() < 0.07:
        return 0
    return 1

def random_mid_chance (my_moves, opp_moves, total_turns_known):
    if random.random() < 0.20:
        return 0
    return 1

def every_fifth (my_moves, opp_moves, total_turns_known):
    if (len(my_moves) % 5 == 0) and len(my_moves) >= 0:
        return 0
    return 1
    
### BONUS STRATEGIES

def tit_for_two_tats (my_moves, opp_moves, total_turns_known):
    # Nice
    if len(opp_moves) <= 1: # first two moves
        return 1
    elif ((opp_moves[-1] == 0) and (opp_moves[-2] == 0)):
        return 0
    return 1



In [401]:
test_game = GameState(name_witheld, tit_for_tat, 200, 0)
test_game.run_game()

Player A <function name_witheld at 0x7c49c2ef16c0> : 505 points
Player B <function tit_for_tat at 0x7c49c2c99e40> : 505 points


In [395]:
print('round_number', '\t', test_game.player_A_strategy.__name__, '\t', test_game.player_B_strategy.__name__)
for i in range(0, 200):
    
    print(i+1, '\t\t', test_game.player_A_move_history[i], '\t\t', test_game.player_B_move_history[i])

# test_game.player_A_move_history
# test_game.player_B_move_history

round_number 	 name_witheld 	 tit_for_tat
1 		 1 		 1
2 		 0 		 1
3 		 1 		 0
4 		 1 		 1
5 		 1 		 1
6 		 1 		 1
7 		 1 		 1
8 		 1 		 1
9 		 1 		 1
10 		 0 		 1
11 		 1 		 0
12 		 0 		 1
13 		 1 		 0
14 		 1 		 1
15 		 1 		 1
16 		 0 		 1
17 		 1 		 0
18 		 1 		 1
19 		 0 		 1
20 		 0 		 0
21 		 0 		 0
22 		 1 		 0
23 		 1 		 1
24 		 0 		 1
25 		 0 		 0
26 		 0 		 0
27 		 1 		 0
28 		 0 		 1
29 		 0 		 0
30 		 0 		 0
31 		 0 		 0
32 		 0 		 0
33 		 1 		 0
34 		 1 		 1
35 		 1 		 1
36 		 1 		 1
37 		 0 		 1
38 		 0 		 0
39 		 1 		 0
40 		 0 		 1
41 		 1 		 0
42 		 1 		 1
43 		 1 		 1
44 		 1 		 1
45 		 0 		 1
46 		 1 		 0
47 		 0 		 1
48 		 0 		 0
49 		 1 		 0
50 		 1 		 1
51 		 0 		 1
52 		 0 		 0
53 		 0 		 0
54 		 1 		 0
55 		 0 		 1
56 		 1 		 0
57 		 0 		 1
58 		 1 		 0
59 		 1 		 1
60 		 0 		 1
61 		 1 		 0
62 		 1 		 1
63 		 0 		 1
64 		 0 		 0
65 		 0 		 0
66 		 1 		 0
67 		 1 		 1
68 		 1 		 1
69 		 0 		 1
70 		 0 		 0
71 		 1 		 0
72 		 1 		 1
73 		 1 		 1
74 		 1 		 1
75 		

In [39]:
test_game.player_A_strategy

<function __main__.tit_for_tat(my_moves, opp_moves, total_turns_known)>